# Clase 18  
Andrea Gutierrez Renteria

## Paso 1. Librerías y parámetros

In [6]:
import numpy as np
from scipy.stats import norm

rng = np.random.default_rng(123)

In [9]:
S0 = 100
K = 100
sigma = 0.2
r = 0.05
T = 1
q = 0
m  = 252         
n  = 100  
dt = T / m   

## Paso 2. Simular las 100 rutas (GBM neutral al riesgo)

In [10]:
Z = rng.standard_normal((n, m))                       # n x m
incrementos = (r - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z
S = S0 * np.exp(np.cumsum(incrementos, axis=1))       # trayectorias t1,...,tm

## Paso 3. Pagos descontados

In [11]:
A = S.mean(axis=1)                       # promedio aritmético por ruta
G = np.exp(np.log(S).mean(axis=1))       # promedio geométrico por ruta

Y = np.exp(-r*T) * np.maximum(A - K, 0)  # variable objetivo
X = np.exp(-r*T) * np.maximum(G - K, 0)  # variable de control

## Paso 4. E[X] exacta (fórmula cerrada de la geométrica)

In [12]:
mu_G = np.log(S0) + (r - 0.5*sigma**2) * T * (m + 1) / (2*m)
v_G  = sigma**2 * T * (m + 1) * (2*m + 1) / (6*m**2)
s_G  = np.sqrt(v_G)

d1 = (mu_G + v_G - np.log(K)) / s_G
d2 = d1 - s_G

EX = np.exp(-r*T) * (np.exp(mu_G + 0.5*v_G) * norm.cdf(d1) - K * norm.cdf(d2))

## Paso 5. Coeficiente b

In [13]:
x_bar, y_bar = X.mean(), Y.mean()

num = np.sum((X - x_bar) * (Y - y_bar))
den = np.sum((X - x_bar)**2)
b_hat = num / den

## Paso 6. Estimador con variable de control

In [14]:
Yb = Y - b_hat * (X - EX)

## Paso 7. Resultados y comparación

In [15]:
rho = np.corrcoef(X, Y)[0, 1]

print(f"E[X] exacta : {EX:.6f}")
print(f"x_bar       : {x_bar:.6f}")
print(f"y_bar       : {y_bar:.6f}")
print(f"b_hat       : {b_hat:.6f}")
print(f"rho_xy      : {rho:.6f}\n")

print(f"{'Estimador':<28}{'Precio':>10}{'Varianza':>12}{'Error est.':>12}")
print(f"{'Estandar (Y)':<28}{Y.mean():>10.4f}{Y.var(ddof=1):>12.4f}{Y.std(ddof=1)/np.sqrt(n):>12.4f}")
print(f"{'Variable de control (Y(b))':<28}{Yb.mean():>10.4f}{Yb.var(ddof=1):>12.4f}{Yb.std(ddof=1)/np.sqrt(n):>12.4f}")

print(f"\nReduccion de varianza: {100*(1 - Yb.var(ddof=1)/Y.var(ddof=1)):.2f} %")
print(f"rho^2                : {100*rho**2:.2f} %")

E[X] exacta : 5.565509
x_bar       : 5.668671
y_bar       : 5.921797
b_hat       : 1.044456
rho_xy      : 0.999515

Estimador                       Precio    Varianza  Error est.
Estandar (Y)                    5.9218     61.3209      0.7831
Variable de control (Y(b))      5.8140      0.0594      0.0244

Reduccion de varianza: 99.90 %
rho^2                : 99.90 %
